# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


In [1]:
import Pkg
Pkg.add("Distributions")
Pkg.add("JuMP")
Pkg.add("MiniZinc")
Pkg.add("MathOptInterface")
Pkg.add("DataStructures")
Pkg.add("Distributions")
Pkg.add("DataFrames")

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\X\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No

## Generowanie plików z problemami

In [2]:
using Distributions

function make_dzn(n::Int, id::Int, R::Int = 100, α::Float64 = 0.5)
    weights = rand(DiscreteUniform(R / 10, R), n)

    # Strong corelation between profits and weights makes dataset hard
    profits = [rand(DiscreteUniform(R / 10, weight)) + R / 10 for weight in weights]
    profits = Array{Int}(profits)

    # capacity is typically some α times sum of weights
    capacity = Int64(round(α * sum(weights)));
    content = """
    ITEM = _(1..$n);
    capacity = $capacity;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_$id.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn (generic function with 3 methods)

In [3]:
n_items = [5, 5, 5, 10, 15, 40]

for (id, i) in enumerate(n_items)
    make_dzn(i, id)
end

In [4]:
# Gecode to g..no (używaj cp-sat albo highsa)

for i in 1:length(n_items)
    result = read(`minizinc --solver cp-sat my_knapsack.mzn knapsack_$i.dzn` , String)
    println(result)
end

taken = [false, true, true, false, false];
profit = 126;
weight = 141;
----------

taken = [false, true, true, false, false];
profit = 80;
weight = 75;
----------

taken = [true, false, true, true, false];
profit = 110;
weight = 99;
----------

taken = [true, false, true, true, false, true, false, true, false, false];
profit = 308;
weight = 337;
----------

taken = [true, true, true, false, false, true, false, true, false, false, true, true, false, true, true];
profit = 411;
weight = 378;
----------

taken = [true, true, true, true, true, false, true, false, true, true, true, false, false, false, false, true, false, false, false, true, false, true, false, true, false, true, false, true, true, true, true, true, false, true, false, true, true, true, true, false];
profit = 1058;
weight = 1007;
----------



## Tabu Search

In [5]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Symbol,Int})
    if move[1] === :add
        return (:remove, move[2])
    else
        return (:add, move[2])
    end
end


function possible_moves(p::KnapsackProblem, x::Vector{Bool})
    move_list = Tuple{Symbol,Int}[]
    current_weight = sum(p.weights .* x)
    # add item
    for i in eachindex(x, p.weights)
        if !x[i] && current_weight + p.weights[i] <= p.capacity
            push!(move_list, (:add, i))
        end
    end
    # remove item
    for i in eachindex(x, p.weights)
        if x[i]
            push!(move_list, (:remove, i))
        end
    end
    return move_list
end



possible_moves (generic function with 1 method)

In [6]:
function read_problem(file_path::String)
    data = Dict{Symbol, Any}()
    
    open(file_path, "r") do file
        for line in eachline(file)
            clean_line = strip(line)
            clean_line = replace(clean_line, ";" => "")
            
            if isempty(clean_line) || startswith(clean_line, "%")
                continue
            end

            parts = split(clean_line, "=", limit=2)
            if length(parts) == 2
                var_name = Symbol(strip(parts[1]))
                var_value_str = strip(parts[2])

                if occursin("..", var_value_str)
                    m = match(r"(\d+)\.\.(\d+)", var_value_str)
                    if m !== nothing
                        start_val = parse(Int, m.captures[1])
                        end_val = parse(Int, m.captures[2])
                        data[var_name] = start_val:end_val
                    end
                else
                    try
                        data[var_name] = eval(Meta.parse(var_value_str))
                    catch e
                        @warn "Could not parse value for $var_name: $var_value_str"
                    end
                end
            end
        end
    end
    
    return KnapsackProblem(data[:capacity], data[:weights], data[:profits])
end

read_problem (generic function with 1 method)

In [7]:
for i in 1:length(n_items)
    result = read(`minizinc --solver cp-sat my_knapsack.mzn knapsack_$i.dzn` , String)
    println("Minizinc:")
    println(result)

    kp = read_problem("knapsack_$i.dzn")

    x0 = fill(false, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)

    println("Tabu Search:")
    println(sol.==1)
    println("Profit: ", -st.best_seen_obj)
    #println("Last iteration: ", st.iter)

    println("---------")
    println("=========")
end

Minizinc:
taken = [false, true, true, false, false];
profit = 126;
weight = 141;
----------

Tabu Search:
Bool[0, 1, 1, 0, 0]
Profit: 126
---------
Minizinc:
taken = [false, true, true, false, false];
profit = 80;
weight = 75;
----------

Tabu Search:
Bool[0, 1, 1, 0, 0]
Profit: 80
---------
Minizinc:
taken = [true, false, true, true, false];
profit = 110;
weight = 99;
----------

Tabu Search:
Bool[1, 0, 1, 1, 0]
Profit: 110
---------
Minizinc:
taken = [true, false, true, true, false, true, false, true, false, false];
profit = 308;
weight = 337;
----------

Tabu Search:
Bool[1, 1, 0, 1, 0, 0, 0, 1, 0, 0]
Profit: 271
---------
Minizinc:
taken = [true, true, true, false, false, true, false, true, false, false, true, true, false, true, true];
profit = 411;
weight = 378;
----------

Tabu Search:
Bool[0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0]
Profit: 352
---------
Minizinc:
taken = [true, true, true, true, true, false, true, false, true, true, true, false, false, false, false, true, fals

In [8]:
using Distributions

function make_3dzn(n::Int, id::Int, R::Int=100, α::Float64=0.5)
    weights = rand(DiscreteUniform(R/10, R), n)
    profits = [rand(DiscreteUniform(R/10, weight)) + R/10 for weight in weights]
    profits = Array{Int}(profits)
    total = sum(weights)
    c1 = Int(round(α * total / 3))
    c2 = Int(round(α * total / 3))
    c3 = Int(round(α * total / 3))
    content = """
ITEM = 1..$n;
capacity1 = $c1;
capacity2 = $c2;
capacity3 = $c3;
profits = $profits;
weights = $weights;
"""
    open("knapsack3_$id.dzn", "w") do f; write(f, content); end
end

n_sets = [10,15,20,25,30,35,40,45,50,60]   # 10 różnych wielkości (zestawów parametrów)
for (id, n) in enumerate(n_sets)
    make_3dzn(n, id)
end
println("Wygenerowano 10 zestawów parametrów (knapsack3_1.dzn … knapsack3_10.dzn)")

Wygenerowano 10 zestawów parametrów (knapsack3_1.dzn … knapsack3_10.dzn)


In [9]:
using DataStructures

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

struct MultiKnapsackProblem
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::MultiKnapsackProblem, x::Vector{Int})
    profit = 0
    for i in eachindex(x)
        if x[i] > 0
            profit += p.profits[i]
        end
    end
    return -profit
end

function apply!(x::Vector{Int}, move::Tuple{Symbol,Int,Int,Int})
    if move[1] === :add;      x[move[2]] = move[3]
    elseif move[1] === :remove; x[move[2]] = 0
    elseif move[1] === :transfer; x[move[2]] = move[4]
    end
    return x
end

function invert_move(::MultiKnapsackProblem, move::Tuple{Symbol,Int,Int,Int})
    if move[1] === :add;      return (:remove, move[2], move[3], 0)
    elseif move[1] === :remove; return (:add, move[2], move[3], 0)
    elseif move[1] === :transfer; return (:transfer, move[2], move[4], move[3])
    end
end

function possible_moves(p::MultiKnapsackProblem, x::Vector{Int}; extended::Bool=false)
    move_list = Tuple{Symbol,Int,Int,Int}[]
    cur_w = zeros(Int, 3)
    
    # Oblicz aktualne obciążenie plecaków
    for i in eachindex(x)
        if x[i] > 0
            cur_w[x[i]] += p.weights[i]
        end
    end

    for i in eachindex(x)
        ck = x[i]  # current knapsack (0 = poza)
        
        for nk in 0:3
            nk == ck && continue
            
            if ck == 0 && nk > 0  # ADD do plecaka nk
                if cur_w[nk] + p.weights[i] <= p.capacities[nk]
                    push!(move_list, (:add, i, nk, 0))
                end
                
            elseif ck > 0 && nk == 0  # REMOVE
                push!(move_list, (:remove, i, ck, 0))
                
            elseif ck > 0 && nk > 0 && extended  # TRANSFER z ck do nk
                if cur_w[nk] + p.weights[i] <= p.capacities[nk]
                    push!(move_list, (:transfer, i, ck, nk))
                end
            end
        end
    end
    return move_list
end

function TabuState(p::MultiKnapsackProblem, x0::Vector{Int}; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    
    # Bezpieczne obliczenie objective nawet gdy nic nie jest wzięte
    obj = objective(p, x0)
    
    return TabuState{eltype(moves), typeof(x0), typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), 
        copy(x0), obj, 
        copy(x0), copy(x0), 
        1
    )
end

function solve_tabu(p, s::TabuState; iteration_limit::Int=100000, extended::Bool=false)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current; extended=extended)
        best_move = 0; best_obj = Inf
        for (i,m) in enumerate(moves)
            in(m, s.tabu_buffer) && continue
            copyto!(s.considered, s.current)
            apply!(s.considered, m)
            val = objective(p, s.considered)
            if val < best_obj
                best_move = i; best_obj = val
            end
        end
        best_move == 0 && break
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_obj < s.best_seen_obj
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_obj
        end
        s.iter += 1
    end
    return s.best_seen
end

solve_tabu (generic function with 1 method)

In [10]:
function read_3problem(file_path::String)
    data = Dict{Symbol, Any}()
    
    open(file_path, "r") do file
        for line in eachline(file)
            clean_line = strip(line)
            clean_line = replace(clean_line, ";" => "")
            
            if isempty(clean_line) || startswith(clean_line, "%")
                continue
            end

            parts = split(clean_line, "=", limit=2)
            if length(parts) == 2
                var_name = Symbol(strip(parts[1]))
                var_value_str = strip(parts[2])

                # Specjalne traktowanie ITEM (range)
                if var_name === :ITEM
                    # Ignorujemy wartość ITEM, bo nie jest nam potrzebna
                    continue
                end

                # Usuwamy ewentualne underscore z range'ów
                var_value_str = replace(var_value_str, r"_\s*\(\s*(\d+)\s*\.\.\s*(\d+)\s*\)" => s"\1:\2")
                var_value_str = replace(var_value_str, r"(\d+)\s*\.\.\s*(\d+)" => s"\1:\2")

                try
                    data[var_name] = eval(Meta.parse(var_value_str))
                catch e
                    @warn "Could not parse value for $var_name: $var_value_str" exception=e
                    # Fallback - spróbuj jako string i potem parse
                    try
                        data[var_name] = Meta.parse(var_value_str)
                    catch
                        data[var_name] = var_value_str
                    end
                end
            end
        end
    end
    
    # Tworzymy strukturę problemu
    return MultiKnapsackProblem(
        [data[:capacity1], data[:capacity2], data[:capacity3]],
        data[:weights],
        data[:profits]
    )
end

read_3problem (generic function with 1 method)

In [11]:
using Distributions

function make_3dzn(n::Int, id::Int, R::Int=100, α::Float64=0.5)
    weights = rand(DiscreteUniform(R/10, R), n)
    profits = [rand(DiscreteUniform(R/10, weight)) + R/10 for weight in weights]
    profits = Array{Int}(profits)
    total = sum(weights)
    c1 = Int(round(α * total / 3))
    c2 = Int(round(α * total / 3))
    c3 = Int(round(α * total / 3))
    content = """
ITEM = 1..$n;
capacity1 = $c1;
capacity2 = $c2;
capacity3 = $c3;
profits = $profits;
weights = $weights;
"""
    open("knapsack3_$id.dzn", "w") do f; write(f, content); end
end

n_sets = [10,15,20,25,30,35,40,45,50,60]   # 10 różnych wielkości (zestawów parametrów)
for (id, n) in enumerate(n_sets)
    make_3dzn(n, id)
end
println("Wygenerowano 10 zestawów parametrów (knapsack3_1.dzn … knapsack3_10.dzn)")

Wygenerowano 10 zestawów parametrów (knapsack3_1.dzn … knapsack3_10.dzn)


W ramach eksperymentów wykonano obliczenia dla 10 instancji problemu z różną liczbą przedmiotów (10–60). 
Dla każdej instancji porównano rozwiązanie dokładne uzyskane za pomocą MiniZinc (solver cp-sat) z dwiema wersjami algorytmu tabu search: podstawową oraz rozszerzoną o ruch przeniesienia przedmiotu między plecakami. Algorytm tabu uruchamiano z trzema różnymi długościami listy zakazów (1, 2, 5), a wyniki uśredniano. Wersję z transferem testowano 10-krotnie dla każdej instancji.

In [12]:
using Statistics, DataFrames, Printf

println("=== PODSUMOWANIE WYNIKÓW DLA TRZECH PLECAKÓW ===\n")

n_sets = [10,15,20,25,30,35,40,45,50,55]
results = []

for (id, n) in enumerate(n_sets)
    println("Zestaw $id (n = $n)")
    kp_file = "knapsack3_$id.dzn"
    kp = read_3problem(kp_file)
    
    # ==================== MINI ZINC ====================
    t_min = @elapsed begin
        res_min = read(`minizinc --solver cp-sat my_3knapsacks.mzn $kp_file`, String)
    end
    m = match(r"total_profit = (\d+)", res_min)
    profit_min = m !== nothing ? parse(Int, m.captures[1]) : 0
    
    println("   MiniZinc     → profit: $profit_min    czas: $(round(t_min, digits=2)) s")
    
    # ==================== TABU BASIC (buf = 1,2,5) ====================
    x0 = zeros(Int, length(kp.weights))
    tabu_profits = Float64[]
    tabu_times   = Float64[]
    
    for buf_len in [1, 2, 5]
        st = TabuState(kp, x0; buffer_length = buf_len)
        t = @elapsed sol = solve_tabu(kp, st; iteration_limit=100000, extended=false)
        push!(tabu_profits, -st.best_seen_obj)
        push!(tabu_times, t)
    end
    
    avg_tabu_profit = mean(tabu_profits)
    avg_tabu_time   = mean(tabu_times)
    
    println("   Tabu basic   → profit: $(round(avg_tabu_profit, digits=0))    czas: $(round(avg_tabu_time, digits=2)) s")
    
    # ==================== TABU + TRANSFER (10 uruchomień) ====================
    transfer_profits = Float64[]
    transfer_times   = Float64[]
    
    for run in 1:10
        st = TabuState(kp, x0; buffer_length = 5)
        t = @elapsed sol = solve_tabu(kp, st; iteration_limit=100000, extended=true)
        push!(transfer_profits, -st.best_seen_obj)
        push!(transfer_times, t)
    end
    
    avg_transfer_profit = mean(transfer_profits)
    avg_transfer_time   = mean(transfer_times)
    
    println("   Tabu+transfer→ profit: $(round(avg_transfer_profit, digits=0))    czas: $(round(avg_transfer_time, digits=2)) s")
    println("   ────────────────────────────────────────────────────────────────")
    
    push!(results, (
        n = n,
        minizinc_profit = profit_min,
        minizinc_time   = round(t_min, digits=2),
        tabu_profit     = round(avg_tabu_profit, digits=0),
        tabu_time       = round(avg_tabu_time, digits=2),
        transfer_profit = round(avg_transfer_profit, digits=0),
        transfer_time   = round(avg_transfer_time, digits=2)
    ))
end

# ====================== ŁADNA TABELKA ======================
df = DataFrame(
    "Liczba przedmiotów (n)"          => [r.n for r in results],
    "MiniZinc profit"                 => [r.minizinc_profit for r in results],
    "MiniZinc czas [s]"               => [r.minizinc_time   for r in results],
    "Tabu basic profit"               => [r.tabu_profit     for r in results],
    "Tabu basic czas [s]"             => [r.tabu_time       for r in results],
    "Tabu+transfer profit"            => [r.transfer_profit for r in results],
    "Tabu+transfer czas [s]"          => [r.transfer_time   for r in results]
)

println("\n")
println("══════════════════════════════════════════════════════════════════════════════")
println("                    PODSUMOWANIE WYNIKÓW – TABELA")
println("══════════════════════════════════════════════════════════════════════════════")
show(df, allrows=true, allcols=true, eltypes=false)

=== PODSUMOWANIE WYNIKÓW DLA TRZECH PLECAKÓW ===

Zestaw 1 (n = 10)
   MiniZinc     → profit: 262    czas: 0.17 s
   Tabu basic   → profit: 262.0    czas: 0.04 s
   Tabu+transfer→ profit: 262.0    czas: 0.04 s
   ────────────────────────────────────────────────────────────────
Zestaw 2 (n = 15)
   MiniZinc     → profit: 432    czas: 0.54 s
   Tabu basic   → profit: 427.0    czas: 0.05 s
   Tabu+transfer→ profit: 416.0    czas: 0.04 s
   ────────────────────────────────────────────────────────────────
Zestaw 3 (n = 20)
   MiniZinc     → profit: 507    czas: 0.88 s
   Tabu basic   → profit: 463.0    czas: 0.05 s
   Tabu+transfer→ profit: 430.0    czas: 0.04 s
   ────────────────────────────────────────────────────────────────
Zestaw 4 (n = 25)
   MiniZinc     → profit: 752    czas: 3.68 s
   Tabu basic   → profit: 728.0    czas: 0.07 s
   Tabu+transfer→ profit: 732.0    czas: 0.07 s
   ────────────────────────────────────────────────────────────────
Zestaw 5 (n = 30)
   MiniZinc     → pr

## Symulowane Wyżarzanie

In [13]:
using DataStructures
using Distributions

mutable struct SAState{P, TF}
    best_seen::P
    best_seen_obj::TF
    current::P
    current_obj::TF
    considered::P
    temp::Float64
    iter::Int
end

function SAState(p, x0; initial_temp::Float64=100.0)
    obj = objective(p, x0)
    return SAState{typeof(x0), typeof(obj)}(
        copy(x0), obj, copy(x0), obj, copy(x0), initial_temp, 1
    )
end

function solve_sa(p, s::SAState; iteration_limit::Int=1000, cooling_rate::Float64=0.95)
    while s.iter < iteration_limit
        move = random_move(p, s.current) 
        
        copyto!(s.considered, s.current)
        apply!(s.considered, move)
        trial_obj = objective(p, s.considered)
    
        ΔE = trial_obj - s.current_obj
        
        if ΔE < 0 || rand() < exp(-ΔE / s.temp)
            copyto!(s.current, s.considered)
            s.current_obj = trial_obj
            
            if s.current_obj < s.best_seen_obj
                copyto!(s.best_seen, s.current)
                s.best_seen_obj = s.current_obj
            end
        end
        
        s.temp *= cooling_rate
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function random_move(p::KnapsackProblem, x::Vector{Bool})
    i = rand(1:length(x))
    
    current_weight = sum(p.weights .* x)
    
    if x[i]
        return (:remove, i)
    else
        if current_weight + p.weights[i] <= p.capacity
            return (:add, i)
        else
            in_bag_indices = findall(x)
            if isempty(in_bag_indices)
                return (:add, i)
            end
            return (:remove, rand(in_bag_indices))
        end
    end
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end



apply! (generic function with 2 methods)

In [14]:
function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(3000, profits, weights)
end

kp1 = generate_problem()


KnapsackProblem(3000, [470, 574, 541, 314, 673, 785, 717, 73, 411, 604  …  708, 542, 45, 404, 746, 488, 662, 635, 408, 348], [70, 48, 100, 49, 58, 64, 96, 42, 28, 63  …  19, 41, 63, 65, 26, 65, 85, 97, 98, 53])

In [15]:
function test_sa(kp)
    x0 = fill(false, length(kp.weights))
    
    st = SAState(kp, x0; initial_temp=1000.0) 
    
    sol = solve_sa(kp, st; iteration_limit=1000000, cooling_rate=0.99999)
    
    println("Items selected (indices): ", findall(sol))
    println("Best objective (Profit): ", -st.best_seen_obj)
    println("Total Weight: ", sum(kp.weights[sol]))
    println("Last iteration: ", st.iter)
end

test_sa (generic function with 1 method)

In [16]:
test_sa(kp1)

Items selected (indices): [8, 12, 20, 22, 26, 27, 30, 36, 56, 63, 64, 69, 72, 78, 84, 86, 93]
Best objective (Profit): 1059
Total Weight: 2990
Last iteration: 1000000
